In [1]:
import pandas as pd
import numpy as np
import os
import sys

from model_metrics import summarize_model_performance
from model_tuner import loadObjects

sys.path.append("../")

from core.functions import (
    normalize_actor,
    build_actor_interaction_graph,
    add_pairwise_embedding_features,
)

/home/lshpaner/Python_Projects/acled_ukraine/acled_venv_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_linear_reg = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/952695909840606430/debe55ab62b54c85b860db9506b1eb2b/artifacts/lr_log_fatalities/model.pkl"
)
model_lasso_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/952695909840606430/b8c16d4d3bdd4c5d8c733c99ba95d9b0/artifacts/lasso_log_fatalities/model.pkl"
)
model_xgb_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/952695909840606430/93a3c1984d264e1ead9897ce6fcc678f/artifacts/xgb_log_fatalities/model.pkl"
)
model_cat_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/952695909840606430/1f9f65deb92d41c7971b2f4c439bf776/artifacts/cat_log_fatalities/model.pkl"
)

Object loaded!
Object loaded!
Object loaded!
Object loaded!


In [3]:
X_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_test.parquet"
)

In [4]:
y_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/y_test_log_fatalities.parquet"
)

In [5]:
pre = model_linear_reg.estimator.named_steps[
    "preprocess_column_transformer_Preprocessor"
]

feature_names = pre.get_feature_names_out()

In [6]:
model_linear_reg.estimator.named_steps

{'preprocess_column_transformer_Preprocessor': ColumnTransformer(transformers=[('num',
                                  Pipeline(steps=[('scaler', StandardScaler()),
                                                  ('imputer', SimpleImputer())]),
                                  ['civilian_targeting', 'latitude', 'longitude',
                                   'percentage_missing']),
                                 ('cat',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(fill_value='missing',
                                                                 strategy='constant')),
                                                  ('encoder',
                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                  ['admin1', 'sub_event_type', 'interaction',
                                   'source_scale', 'geo_precision'])]),
 'lr': LinearRegression(n_

In [7]:
regression_metrics = summarize_model_performance(
    model=[
        model_linear_reg,
        model_lasso_rfe,
        model_xgb_rfe,
        model_cat_rfe,
    ],
    model_title=[
        "Linear Regression",
        "Lasso RFE",
        "XGBoost RFE",
        "CatBoost RFE",
    ],
    X=X_test,
    y=y_test,
    model_type="regression",
    include_adjusted_r2=True,
    return_df=True,
    decimal_places=2,
    overall_only=False,
)

/home/lshpaner/Python_Projects/acled_ukraine/acled_venv_311/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning:

[01:14:35] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.




In [8]:
regression_metrics = regression_metrics.drop(
    columns=["Metric", "Variable", "Coefficient"]
)

In [9]:
regression_metrics

,Model,MAE,MAPE,MSE,RMSE,Expl. Var.,R^2,Adj. R^2
0,Linear Regression,0.35,46.71,0.43,0.66,0.24,0.24,0.24
1,Lasso RFE,0.37,58.60,0.47,0.68,0.20,0.18,0.18
2,XGBoost RFE,0.26,50.52,0.38,0.62,0.36,0.33,0.33
3,CatBoost RFE,0.28,52.01,0.39,0.62,0.35,0.32,0.32


In [ ]:
len(model_linear_reg.get_feature_names())

In [ ]:
len(model_xgb_rfe.get_feature_names())

In [ ]:
coef_df = pd.DataFrame(
    {
        "feature": model_linear_reg.get_feature_names(),
        "coef": model_linear_reg.estimator.named_steps["lr"].coef_,
    }
).sort_values("coef", key=abs, ascending=False)

coef_df.head(20)

In [ ]:
model_xgb_rfe.estimator

In [ ]:
df = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/raw/acled_ukraine_data_2026_01_02.parquet"
)

In [ ]:
X_test = X_test.assign(
    **pd.get_dummies(X_test[["sub_event_type", "source_scale"]], dtype=int)
)

In [ ]:
[col for col in X_test.columns if "Regional" in col]

In [ ]:
[col for col in X_test.columns if "source" in col]

In [ ]:
[col for col in model_xgb_rfe.get_feature_names() if "source" in col]

In [ ]:
import os

os.getcwd()

In [ ]:
os.path.exists("../data")

In [ ]:
from model_metrics import plot_3d_pdp

plot_3d_pdp(
    model=model_xgb_rfe.estimator,
    dataframe=X_test,
    feature_names=["sub_event_type", "source_scale"],
    x_label="sub_event_type",
    y_label="source_scale",
    z_label="Partial Dependence",
    title="3D Partial Dependence Plot of sub_event_type vs. source_scale",
    html_file_path="../data",
    # image_filename="3d_pdp",
    html_file_name="3d_pdp.html",
    image_path_svg="../data",
    plot_type="interactive",
    text_wrap=80,
    zoom_out_factor=1.2,
    grid_resolution=30,
    label_fontsize=8,
    tick_fontsize=6,
    title_x=0.38,
    top_margin=20,
    right_margin=80,
    left_margin=70,
    cbar_x=0.9,
    cbar_thickness=25,
    show_modebar=False,
    enable_zoom=True,
    save_plots="html",
)

In [ ]:
from model_metrics import summarize_model_performance

In [ ]:
y_test.rename(columns={"log_fatalities": "actual_log_fatalities"}, inplace=True)

In [ ]:
y_test

In [ ]:
X_test.head()

In [ ]:
predictions_x_test = pd.Series(
    model_xgb.predict(X_test), index=X_test.index, name="prediction"
)

In [ ]:
log_preds = predictions_x_test.to_frame(name="log_pred_fatalities")

In [ ]:
log_preds["pred_fatalities"] = round(
    np.expm1(log_preds["log_pred_fatalities"]), 0
).astype(int)

In [ ]:
log_preds_merge = log_preds.join(X_test, how="inner", on="event_id_cnty")

In [ ]:
y_test_actual = pd.DataFrame(
    np.expm1(y_test["actual_log_fatalities"]).round(0).astype(int),
    columns=["actual_log_fatalities"],
).rename(columns={"actual_log_fatalities": "actual_fatalities"})

In [ ]:
y_test_actual

In [ ]:
log_preds_merge = y_test_actual.merge(log_preds_merge, how="inner", on="event_id_cnty")

In [ ]:
log_preds_merge[["actual_fatalities", "pred_fatalities"]]

In [ ]:
from sklearn.metrics import r2_score

r2_score(log_preds_merge["actual_fatalities"], log_preds_merge["pred_fatalities"])

In [ ]:
from eda_toolkit import scatter_fit_plot

scatter_fit_plot(
    df=log_preds_merge,
    x_vars=["actual_fatalities"],
    y_vars=["pred_fatalities"],
    show_legend=True,
    show_plot="subplots",
    subplot_figsize=None,
    label_fontsize=14,
    tick_fontsize=12,
    add_best_fit_line=True,
    scatter_color="#808080",
    show_correlation=True,
)

In [ ]:
log_preds_merge

In [ ]:
log_preds_merge.to_csv(
    os.path.join("..", "data", "processed", "log_preds_merge.csv"), index=False
)

In [ ]:
model_xgb.get_feature_names()

In [ ]:
dir(model_xgb)